# pandas Complete Notebook


This notebook is a practical learning and reference guide for pandas, aimed at ML engineers and data-oriented software engineers. It focuses on the pandas skills that show up in real workflows: loading data, inspecting it, cleaning it, joining it, reshaping it, and preparing model-ready tables.


## 1. Introduction to pandas

pandas is built for labeled tabular data. Where NumPy gives you fast n-dimensional arrays, pandas adds row labels, column labels, missing-value handling, mixed column types, and higher-level tools for working with datasets.

The two core objects in pandas are:

### Series
A one-dimensional labeled array.

### DataFrame
A two-dimensional labeled table made of columns, where each column can have its own dtype.

For ML and data work, pandas is usually the tool you reach for when you are:
- loading tabular data from files
- cleaning messy columns
- filtering and grouping records
- creating features from timestamps, strings, and categories
- preparing `X` and `y` before handing them to NumPy or a modeling library

NumPy is still the better tool for:
- pure numerical tensor work
- broadcasting-heavy math
- lower-level array manipulation
- performance-sensitive numeric kernels


In [1]:
import pandas as pd
import numpy as np

simple_series = pd.Series([10, 20, 30], index=["a", "b", "c"], name="score")
simple_df = pd.DataFrame(
    {
        "employee": ["Ana", "Ben", "Cara"],
        "score": [88, 92, 85],
        "team": ["A", "A", "B"],
    },
    index=["row_1", "row_2", "row_3"],
)

print("\nsimple_series:\n", simple_series)
print("\nsimple_df:\n", simple_df)



simple_series:
 a    10
b    20
c    30
Name: score, dtype: int64

simple_df:
       employee  score team
row_1      Ana     88    A
row_2      Ben     92    A
row_3     Cara     85    B


## 2. Setup and reusable example datasets

I think reusing the same datasets across sections builds intuition much better than constantly inventing new examples. So, the notebook uses a small set of core tables

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

employees_df = pd.DataFrame(
    {
        "employee_id": [101, 102, 103, 104, 105, 106],
        "name": ["Ana", "Ben", "Cara", "Dan", "Eli", "Fay"],
        "department": ["Data", "Data", "Sales", "Sales", "Ops", "Ops"],
        "city": ["Berlin", "Berlin", "Paris", "Paris", "Rome", "Rome"],
        "salary": [78000, 82000, 65000, 67000, np.nan, 61000],
        "performance_score": [4.7, 4.3, 4.1, np.nan, 3.8, 4.0],
        "years_experience": [6, 8, 4, 5, 7, 3],
        "is_manager": [True, False, False, True, False, False],
        "start_date": pd.to_datetime(
            ["2019-01-15", "2017-07-01", "2021-03-10", "2020-11-05", "2018-05-20", "2022-08-14"]
        ),
    }
)

sales_df = pd.DataFrame(
    {
        "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
        "customer_id": [1, 2, 1, 3, 4, 2, 5, 3],
        "product": ["Laptop", "Mouse", "Keyboard", "Laptop", "Mouse", "Monitor", "Keyboard", "Monitor"],
        "channel": ["web", "store", "web", "partner", "store", "web", "web", "partner"],
        "amount": [1200, 25, 80, 1250, 30, 300, 85, 320],
        "quantity": [1, 1, 1, 1, 2, 1, 1, 1],
        "order_date": pd.to_datetime(
            ["2025-01-03", "2025-01-04", "2025-01-10", "2025-01-15", "2025-02-02", "2025-02-08", "2025-02-14", "2025-02-20"]
        ),
    }
)

messy_df = pd.DataFrame(
    {
        "raw_name": ["  alice  ", "BOB", "Carla ", None, "david"],
        "raw_city": [" berlin", "PARIS ", "Rome", "rome ", " BERLIN "],
        "raw_segment": ["Enterprise", "enterprise", "SMB", " smb ", "Mid-Market"],
        "revenue_text": ["1000", "2,500", None, "850", "1_200"],
        "signup_text": ["2025-01-10", "01/11/2025", "2025.01.12", None, "2025-01-14"],
    }
)

metrics_df = pd.DataFrame(
    {
        "date": pd.date_range("2025-03-01", periods=10, freq="D"),
        "visits": [120, 135, 128, 150, 165, 160, 172, 180, 175, 190],
        "signups": [8, 10, 9, 11, 14, 13, 15, 16, 14, 18],
    }
)

ml_df = pd.DataFrame(
    {
        "customer_id": [1, 2, 3, 4, 5, 6],
        "age": [25, 34, 29, np.nan, 42, 37],
        "income": [42000, 68000, 52000, 61000, 88000, 73000],
        "city": ["Berlin", "Paris", "Berlin", "Rome", "Paris", None],
        "plan": ["basic", "pro", "basic", "enterprise", "pro", "basic"],
        "signup_date": pd.to_datetime(
            ["2024-05-01", "2024-06-14", "2024-07-20", "2024-08-03", "2024-09-15", "2024-10-01"]
        ),
        "churned": [0, 1, 0, 0, 1, 0],
    }
)

feature_a_df = pd.DataFrame(
    {
        "customer_id": [1, 2, 3, 4],
        "region": ["DE", "FR", "DE", "IT"],
        "credit_score": [710, 690, 730, 640],
    }
)

feature_b_df = pd.DataFrame(
    {
        "customer_id": [1, 1, 2, 5],
        "last_order_amount": [1200, 80, 300, 85],
        "channel": ["web", "web", "web", "web"],
    }
)

print("\nemployees_df:\n", employees_df.head())
print("\nsales_df:\n", sales_df.head())
print("\nmessy_df:\n", messy_df.head())
print("\nmetrics_df:\n", metrics_df.head())
print("\nml_df:\n", ml_df.head())



employees_df:
    employee_id  name department    city   salary  performance_score  years_experience  is_manager start_date
0          101   Ana       Data  Berlin  78000.0                4.7                 6        True 2019-01-15
1          102   Ben       Data  Berlin  82000.0                4.3                 8       False 2017-07-01
2          103  Cara      Sales   Paris  65000.0                4.1                 4       False 2021-03-10
3          104   Dan      Sales   Paris  67000.0                NaN                 5        True 2020-11-05
4          105   Eli        Ops    Rome      NaN                3.8                 7       False 2018-05-20

sales_df:
    order_id  customer_id   product  channel  amount  quantity order_date
0      1001            1    Laptop      web    1200         1 2025-01-03
1      1002            2     Mouse    store      25         1 2025-01-04
2      1003            1  Keyboard      web      80         1 2025-01-10
3      1004            3  

## 3. Creating Series and DataFrames

The most common creation patterns are from lists, dictionaries, records, and NumPy arrays. Checking data types early is usually a good practice.

In [3]:
series_from_list = pd.Series([1.5, 2.0, 3.5], name="metric")
series_from_dict = pd.Series({"berlin": 3, "paris": 5, "rome": 2}, name="office_count")

df_from_dict = pd.DataFrame(
    {
        "product": ["Laptop", "Mouse", "Keyboard"],
        "price": [1200, 25, 80],
        "in_stock": [True, True, False],
    }
)

df_from_records = pd.DataFrame(
    [
        {"customer_id": 1, "plan": "basic", "churned": 0},
        {"customer_id": 2, "plan": "pro", "churned": 1},
        {"customer_id": 3, "plan": "basic", "churned": 0},
    ]
)

array = np.array([[1, 2], [3, 4], [5, 6]])
df_from_array = pd.DataFrame(array, columns=["feature_1", "feature_2"], index=["r1", "r2", "r3"])
df_from_array["bias"] = 1

print("\nseries_from_list:\n", series_from_list)
print("\nseries_from_dict:\n", series_from_dict)
print("\ndf_from_dict.dtypes:\n", df_from_dict.dtypes)
print("\ndf_from_records:\n", df_from_records)
print("\ndf_from_array:\n", df_from_array)



series_from_list:
 0    1.5
1    2.0
2    3.5
Name: metric, dtype: float64

series_from_dict:
 berlin    3
paris     5
rome      2
Name: office_count, dtype: int64

df_from_dict.dtypes:
 product     object
price        int64
in_stock      bool
dtype: object

df_from_records:
    customer_id   plan  churned
0            1  basic        0
1            2    pro        1
2            3  basic        0

df_from_array:
     feature_1  feature_2  bias
r1          1          2     1
r2          3          4     1
r3          5          6     1


## 4. Reading and writing data

In practice, loading data well matters almost as much as analyzing it well. A few import parameters can save a lot of cleanup work: `usecols`, `nrows`, `dtype`, `parse_dates`, `na_values`, and `index_col`.

We may also use sql for extracting data from a data base in production environments.


In [4]:
from io import StringIO

csv_text = StringIO(
    """customer_id,age,income,city,signup_date
1,25,42000,Berlin,2024-05-01
2,34,68000,Paris,2024-06-14
3,,52000,Berlin,2024-07-20
4,NA,61000,Rome,2024-08-03
"""
)

loaded_csv = pd.read_csv(
    csv_text,
    usecols=["customer_id", "age", "income", "city", "signup_date"],
    dtype={"customer_id": "int64", "city": "string"},
    parse_dates=["signup_date"],
    na_values=["NA"],
)

csv_export_preview = loaded_csv.to_csv(index=False)
json_preview = ml_df[["customer_id", "plan", "churned"]].to_json(orient="records")
json_loaded = pd.read_json(StringIO(json_preview))

print("\nloaded_csv:\n", loaded_csv)
print("\njson_loaded:\n", json_loaded)
print("\ncsv_export_preview[:120]:\n", csv_export_preview[:120])



loaded_csv:
    customer_id   age  income    city signup_date
0            1  25.0   42000  Berlin  2024-05-01
1            2  34.0   68000   Paris  2024-06-14
2            3   NaN   52000  Berlin  2024-07-20
3            4   NaN   61000    Rome  2024-08-03

json_loaded:
    customer_id        plan  churned
0            1       basic        0
1            2         pro        1
2            3       basic        0
3            4  enterprise        0
4            5         pro        1
5            6       basic        0

csv_export_preview[:120]:
 customer_id,age,income,city,signup_date
1,25.0,42000,Berlin,2024-05-01
2,34.0,68000,Paris,2024-06-14
3,,52000,Berlin,


## 5. Inspecting and understanding data

Inspection is the first real step in almost every workflow. The goals are to answer: what columns exist, what shape the dataset has, which dtypes were inferred, how many missing values are present, and what representative rows look like.


In [5]:
inspection_summary = {
    "shape": employees_df.shape,
    "columns": employees_df.columns.tolist(),
    "index_type": type(employees_df.index).__name__,
    "dtypes": employees_df.dtypes.astype(str).to_dict(),
    "null_counts": employees_df.isna().sum().to_dict(),
}

print("\ninspection_summary:\n", inspection_summary)



inspection_summary:
 {'shape': (6, 9), 'columns': ['employee_id', 'name', 'department', 'city', 'salary', 'performance_score', 'years_experience', 'is_manager', 'start_date'], 'index_type': 'RangeIndex', 'dtypes': {'employee_id': 'int64', 'name': 'object', 'department': 'object', 'city': 'object', 'salary': 'float64', 'performance_score': 'float64', 'years_experience': 'int64', 'is_manager': 'bool', 'start_date': 'datetime64[ns]'}, 'null_counts': {'employee_id': 0, 'name': 0, 'department': 0, 'city': 0, 'salary': 1, 'performance_score': 1, 'years_experience': 0, 'is_manager': 0, 'start_date': 0}}


In [6]:
employees_preview = employees_df.head(3)
employees_info = employees_df.info()
department_counts = employees_df["department"].value_counts()
city_unique = employees_df["city"].unique()
salary_description = employees_df["salary"].describe()

print("\nemployees_preview:\n", employees_preview )
print("\nemployees_info:\n", employees_info )
print("\ndepartment_counts:\n", department_counts )
print("\ncity_unique:\n", city_unique )
print("\nsalary_description:\n", salary_description )

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   employee_id        6 non-null      int64         
 1   name               6 non-null      object        
 2   department         6 non-null      object        
 3   city               6 non-null      object        
 4   salary             5 non-null      float64       
 5   performance_score  5 non-null      float64       
 6   years_experience   6 non-null      int64         
 7   is_manager         6 non-null      bool          
 8   start_date         6 non-null      datetime64[ns]
dtypes: bool(1), datetime64[ns](1), float64(2), int64(2), object(3)
memory usage: 522.0+ bytes

employees_preview:
    employee_id  name department    city   salary  performance_score  years_experience  is_manager start_date
0          101   Ana       Data  Berlin  78000.0                4.7   

## 6. Selecting columns and rows

Use explicit selection. Attribute access like `df.salary` can work, but it is fragile. The main selection tools are `df["col"]` (gives pandas series), `df[[...]]` (gives a data frame), `.loc`, `.iloc`, `.at`, and `.iat`.


In [7]:
single_col = employees_df["salary"]
multi_col = employees_df[["name", "department", "salary"]]

loc_selection = employees_df.loc[employees_df["department"] == "Data", ["name", "salary"]]
iloc_selection = employees_df.iloc[:3, [1, 2, 4]]
scalar_label = employees_df.at[0, "name"]
scalar_position = employees_df.iat[0, 1]

print("\nsingle_col:\n", single_col )
print("\nmulti_col:\n", multi_col )
print("\nloc_selection:\n", loc_selection )
print("\niloc_selection:\n", iloc_selection )
print("\nscalar_label:\n", scalar_label )
print("\nscalar_position:\n", scalar_position )



single_col:
 0    78000.0
1    82000.0
2    65000.0
3    67000.0
4        NaN
5    61000.0
Name: salary, dtype: float64

multi_col:
    name department   salary
0   Ana       Data  78000.0
1   Ben       Data  82000.0
2  Cara      Sales  65000.0
3   Dan      Sales  67000.0
4   Eli        Ops      NaN
5   Fay        Ops  61000.0

loc_selection:
   name   salary
0  Ana  78000.0
1  Ben  82000.0

iloc_selection:
    name department   salary
0   Ana       Data  78000.0
1   Ben       Data  82000.0
2  Cara      Sales  65000.0

scalar_label:
 Ana

scalar_position:
 Ana


## 7. Indexing fundamentals and alignment

pandas aligns on labels. This is powerful, but it can also surprise you if you mentally expect pure positional behavior. A classic gotcha is arithmetic between two Series with different labels.


In [8]:
employees_by_id = employees_df.set_index("employee_id")
reset_back = employees_by_id.reset_index()
sorted_indexed = employees_by_id.sort_index()

q1 = pd.Series([100, 200, 300], index=["A", "B", "C"], name="q1")
q2 = pd.Series([10, 20, 30], index=["B", "C", "D"], name="q2")
aligned_sum = q1 + q2

print("\nemployees_by_id:\n", employees_by_id.head())
print("\nreset_back:\n", reset_back.head())
print("\nsorted_indexed:\n", sorted_indexed.head())
print("\naligned_sum:\n", aligned_sum)



employees_by_id:
              name department    city   salary  performance_score  years_experience  is_manager start_date
employee_id                                                                                              
101           Ana       Data  Berlin  78000.0                4.7                 6        True 2019-01-15
102           Ben       Data  Berlin  82000.0                4.3                 8       False 2017-07-01
103          Cara      Sales   Paris  65000.0                4.1                 4       False 2021-03-10
104           Dan      Sales   Paris  67000.0                NaN                 5        True 2020-11-05
105           Eli        Ops    Rome      NaN                3.8                 7       False 2018-05-20

reset_back:
    employee_id  name department    city   salary  performance_score  years_experience  is_manager start_date
0          101   Ana       Data  Berlin  78000.0                4.7                 6        True 2019-01-15
1      

## 8. Boolean filtering and conditional logic

Boolean filtering is one of the most-used pandas skills. Keep these habits: wrap each condition in parentheses when using `&` or `|`, use `isin` for membership checks, use `between` for ranges, and use `isna` or `notna` for missing-value filters.


In [9]:
high_salary = employees_df[employees_df["salary"] > 70000]

data_or_ops = employees_df[
    (employees_df["department"].isin(["Data", "Ops"])) &
    (employees_df["years_experience"].between(3, 7))
]

missing_salary = employees_df[employees_df["salary"].isna()]
known_city = ml_df[ml_df["city"].notna()]
query_example = sales_df.query("amount >= 300 and channel != 'store'")
where_example = employees_df["salary"].where(employees_df["salary"] >= 65000, other=65000)
mask_example = employees_df["department"].mask(employees_df["department"] == "Ops", other="Operations")

print("\nhigh_salary:\n", high_salary)
print("\ndata_or_ops:\n", data_or_ops)
print("\nmissing_salary:\n", missing_salary)
print("\nknown_city:\n", known_city)
print("\nquery_example:\n", query_example)
print("\nwhere_example:\n", where_example)
print("\nmask_example:\n", mask_example)



high_salary:
    employee_id name department    city   salary  performance_score  years_experience  is_manager start_date
0          101  Ana       Data  Berlin  78000.0                4.7                 6        True 2019-01-15
1          102  Ben       Data  Berlin  82000.0                4.3                 8       False 2017-07-01

data_or_ops:
    employee_id name department    city   salary  performance_score  years_experience  is_manager start_date
0          101  Ana       Data  Berlin  78000.0                4.7                 6        True 2019-01-15
4          105  Eli        Ops    Rome      NaN                3.8                 7       False 2018-05-20
5          106  Fay        Ops    Rome  61000.0                4.0                 3       False 2022-08-14

missing_salary:
    employee_id name department  city  salary  performance_score  years_experience  is_manager start_date
4          105  Eli        Ops  Rome     NaN                3.8                 7       Fal

## 9. Assignment and column modification

Column creation is where a lot of pandas work happens. Prefer explicit, readable patterns. ⚠️ Avoid chained assignment patterns like `df[df["x"] > 0]["y"] = ...` because they can update a temporary view instead of the original DataFrame.


In [10]:
employees_enriched = employees_df.copy()

employees_enriched["salary_band"] = np.where(
    employees_enriched["salary"] >= 75000, "high",
    np.where(employees_enriched["salary"] >= 65000, "mid", "entry")
)
employees_enriched.loc[employees_enriched["salary"].isna(), "salary_band"] = "unknown"
employees_enriched["department_code"] = employees_enriched["department"].map({"Data": "D", "Sales": "S", "Ops": "O"})

employees_assigned = employees_enriched.assign(
    tenure_years=lambda df: (pd.Timestamp("2025-04-01") - df["start_date"]).dt.days / 365.25,
    compensation_k=lambda df: df["salary"] / 1000,
)

employees_assigned.insert(2, "country", ["DE", "DE", "FR", "FR", "IT", "IT"])
renamed_columns = employees_assigned.rename(columns={"performance_score": "perf_score"})
replaced_segments = messy_df["raw_segment"].replace({" smb ": "SMB", "enterprise": "Enterprise"})

print("\nemployees_assigned:\n", employees_assigned.head())
print("\nrenamed_columns.columns.tolist():\n", renamed_columns.columns.tolist())
print("\nreplaced_segments:\n", replaced_segments)



employees_assigned:
    employee_id  name country department    city   salary  performance_score  years_experience  is_manager start_date  \
0          101   Ana      DE       Data  Berlin  78000.0                4.7                 6        True 2019-01-15   
1          102   Ben      DE       Data  Berlin  82000.0                4.3                 8       False 2017-07-01   
2          103  Cara      FR      Sales   Paris  65000.0                4.1                 4       False 2021-03-10   
3          104   Dan      FR      Sales   Paris  67000.0                NaN                 5        True 2020-11-05   
4          105   Eli      IT        Ops    Rome      NaN                3.8                 7       False 2018-05-20   

  salary_band department_code  tenure_years  compensation_k  
0        high               D      6.209446            78.0  
1        high               D      7.750856            82.0  
2         mid               S      4.060233            65.0  
3        

Human-friendly column management habit: keep feature tables explicit. Drop what you do not need, and `pop` when you want to remove and keep a column in one step.

In [11]:
column_mgmt = employees_enriched.copy()

# drop columns not needed for next step
compact = column_mgmt.drop(columns=["start_date", "is_manager"])

# pop removes and returns the column
salary_series = compact.pop("salary")

print("compact columns:", compact.columns.tolist())
print("popped salary head:")
print(salary_series.head())

compact columns: ['employee_id', 'name', 'department', 'city', 'performance_score', 'years_experience', 'salary_band', 'department_code']
popped salary head:
0    78000.0
1    82000.0
2    65000.0
3    67000.0
4        NaN
Name: salary, dtype: float64


## 10. Missing data handling

Missing values are normal. The important question is not how to remove NaNs, but what the right policy is for each column. A model-ready table usually needs a deliberate missing-data strategy.


In [12]:
ml_missing = ml_df.copy()
null_profile = ml_missing.isna().sum()

ml_missing["age"] = ml_missing["age"].fillna(ml_missing["age"].median())
ml_missing["city"] = ml_missing["city"].fillna("Unknown")

employees_filled = employees_df.copy()
employees_filled["salary"] = employees_filled["salary"].fillna(employees_filled["salary"].median())
employees_filled["performance_score"] = employees_filled["performance_score"].ffill()

dropna_example = messy_df.dropna(subset=["raw_name"])
backfill_example = pd.Series([1, np.nan, np.nan, 4]).bfill()

print("\nnull_profile:\n", null_profile)
print("\nml_missing:\n", ml_missing)
print("employees_filled[[\"name\", \"salary\", \"performance_score\"]]:\n", employees_filled[["name", "salary", "performance_score"]])
print("\ndropna_example:\n", dropna_example)
print("\nbackfill_example:\n", backfill_example)



null_profile:
 customer_id    0
age            1
income         0
city           1
plan           0
signup_date    0
churned        0
dtype: int64

ml_missing:
    customer_id   age  income     city        plan signup_date  churned
0            1  25.0   42000   Berlin       basic  2024-05-01        0
1            2  34.0   68000    Paris         pro  2024-06-14        1
2            3  29.0   52000   Berlin       basic  2024-07-20        0
3            4  34.0   61000     Rome  enterprise  2024-08-03        0
4            5  42.0   88000    Paris         pro  2024-09-15        1
5            6  37.0   73000  Unknown       basic  2024-10-01        0
employees_filled[["name", "salary", "performance_score"]]:
    name   salary  performance_score
0   Ana  78000.0                4.7
1   Ben  82000.0                4.3
2  Cara  65000.0                4.1
3   Dan  67000.0                4.1
4   Eli  67000.0                3.8
5   Fay  61000.0                4.0

dropna_example:
     raw_nam

## 11. Type conversion and dtype management

Dtypes affect correctness, missing-value behavior, memory usage, and compatibility with downstream ML libraries. Common cleanup tasks include `to_numeric`, `to_datetime`, and converting repeated labels to `category`.


In [13]:
typed_df = messy_df.copy()

typed_df["revenue_clean"] = (
    typed_df["revenue_text"]
    .str.replace(",", "")
    .str.replace("_", "")
)
typed_df["revenue"] = pd.to_numeric(typed_df["revenue_clean"], errors="coerce")
typed_df["signup_date"] = pd.to_datetime(typed_df["signup_text"], errors="coerce")
typed_df["raw_city"] = typed_df["raw_city"].astype("string")
typed_df["segment_category"] = typed_df["raw_segment"].str.strip().str.lower().astype("category")

dtype_report = typed_df.dtypes.astype(str)
memory_before = messy_df.memory_usage(deep=True).sum()
memory_after = typed_df[["segment_category"]].memory_usage(deep=True).sum()

print("\ndtype_report:\n", dtype_report)
print("\nmemory_before:\n", memory_before)
print("\nmemory_after:\n", memory_after)
print("\ntyped_df:\n", typed_df)



dtype_report:
 raw_name                    object
raw_city                    string
raw_segment                 object
revenue_text                object
signup_text                 object
revenue_clean               object
revenue                    float64
signup_date         datetime64[ns]
segment_category          category
dtype: object

memory_before:
 1430

memory_after:
 415

typed_df:
     raw_name  raw_city raw_segment revenue_text signup_text revenue_clean  revenue signup_date segment_category
0    alice      berlin  Enterprise         1000  2025-01-10          1000   1000.0  2025-01-10       enterprise
1        BOB    PARIS   enterprise        2,500  01/11/2025          2500   2500.0         NaT       enterprise
2     Carla       Rome         SMB         None  2025.01.12          None      NaN         NaT              smb
3       None     rome         smb           850        None           850    850.0         NaT              smb
4      david   BERLIN   Mid-Market       

## 12. Sorting, ranking, and ordering

Sorting helps you inspect extremes, build reports, and prepare data for window operations. `rank`, `nlargest`, and `nsmallest` are especially useful when you care about top or bottom records.


In [14]:
sorted_employees = employees_df.sort_values(["department", "salary"], ascending=[True, False])
salary_rank = employees_df["salary"].rank(ascending=False, method="dense")
top_sales = sales_df.nlargest(3, "amount")
smallest_sales = sales_df.nsmallest(2, "amount")

print("\nsorted_employees:\n", sorted_employees)
print("\nsalary_rank:\n", salary_rank)
print("\ntop_sales:\n", top_sales)
print("\nsmallest_sales:\n", smallest_sales)



sorted_employees:
    employee_id  name department    city   salary  performance_score  years_experience  is_manager start_date
1          102   Ben       Data  Berlin  82000.0                4.3                 8       False 2017-07-01
0          101   Ana       Data  Berlin  78000.0                4.7                 6        True 2019-01-15
5          106   Fay        Ops    Rome  61000.0                4.0                 3       False 2022-08-14
4          105   Eli        Ops    Rome      NaN                3.8                 7       False 2018-05-20
3          104   Dan      Sales   Paris  67000.0                NaN                 5        True 2020-11-05
2          103  Cara      Sales   Paris  65000.0                4.1                 4       False 2021-03-10

salary_rank:
 0    2.0
1    1.0
2    4.0
3    3.0
4    NaN
5    5.0
Name: salary, dtype: float64

top_sales:
    order_id  customer_id  product  channel  amount  quantity order_date
3      1004            3   Laptop 

## 13. Aggregation and descriptive statistics

Aggregations summarize a dataset into something decision-friendly. Useful tools include `sum`, `mean`, `median`, `min`, `max`, `count`, `quantile`, and `agg`.


In [15]:
numeric_summary = employees_df[["salary", "performance_score", "years_experience"]].agg(["mean", "median", "min", "max"])
sales_quantiles = sales_df["amount"].quantile([0.25, 0.5, 0.75])
rowwise_total = sales_df[["amount", "quantity"]].sum(axis=1).head()
custom_agg = sales_df.agg(
    total_revenue=("amount", "sum"),
    avg_revenue=("amount", "mean"),
    unique_products=("product", "nunique"),
)

print("\nnumeric_summary:\n", numeric_summary)
print("\nsales_quantiles:\n", sales_quantiles)
print("\nrowwise_total:\n", rowwise_total)
print("\ncustom_agg:\n", custom_agg)



numeric_summary:
          salary  performance_score  years_experience
mean    70600.0               4.18               5.5
median  67000.0               4.10               5.5
min     61000.0               3.80               3.0
max     82000.0               4.70               8.0

sales_quantiles:
 0.25     67.5
0.50    192.5
0.75    540.0
Name: amount, dtype: float64

rowwise_total:
 0    1201
1      26
2      81
3    1251
4      32
dtype: int64

custom_agg:
                   amount  product
total_revenue    3290.00      NaN
avg_revenue       411.25      NaN
unique_products      NaN      4.0


`crosstab` is a fast QA tool for category relationships, and `cut` / `qcut` are practical for segmentation-style features.

In [16]:
# crosstab for category relationship checks
channel_product_ct = pd.crosstab(sales_df["channel"], sales_df["product"], margins=True)
print(channel_product_ct)

# binning continuous values
employees_df["salary_bins"] = pd.cut(employees_df["salary"], bins=[0, 65000, 75000, 90000], labels=["low", "mid", "high"])
employees_df["exp_quantiles"] = pd.qcut(employees_df["years_experience"], q=3, labels=["junior", "mid", "senior"])

print("\nemployees_df:\n",employees_df)


product  Keyboard  Laptop  Monitor  Mouse  All
channel                                       
partner         0       1        1      0    2
store           0       0        0      2    2
web             2       1        1      0    4
All             2       2        2      2    8

employees_df:
    employee_id  name department    city   salary  performance_score  years_experience  is_manager start_date  \
0          101   Ana       Data  Berlin  78000.0                4.7                 6        True 2019-01-15   
1          102   Ben       Data  Berlin  82000.0                4.3                 8       False 2017-07-01   
2          103  Cara      Sales   Paris  65000.0                4.1                 4       False 2021-03-10   
3          104   Dan      Sales   Paris  67000.0                NaN                 5        True 2020-11-05   
4          105   Eli        Ops    Rome      NaN                3.8                 7       False 2018-05-20   
5          106   Fay        Op

## 14. GroupBy fundamentals

GroupBy follows **split → apply → combine**. `agg` returns one or a few rows per group, `transform` returns one value per original row aligned to the source data, and `apply` is flexible but often slower and less predictable. ⚠️ Reach for `agg` or `transform` first.


In [ ]:
dept_summary = employees_df.groupby("department").agg(
    avg_salary=("salary", "mean"),
    avg_perf=("performance_score", "mean"),
    headcount=("employee_id", "size"),
)

city_dept_summary = employees_df.groupby(["city", "department"]).agg(
    avg_salary=("salary", "mean"),
    total_experience=("years_experience", "sum"),
).sort_index()

sales_with_group_features = sales_df.copy()
sales_with_group_features["channel_avg_amount"] = sales_with_group_features.groupby("channel")["amount"].transform("mean")
sales_with_group_features["amount_vs_channel_avg"] = sales_with_group_features["amount"] - sales_with_group_features["channel_avg_amount"]

large_group_only = sales_df.groupby("channel").filter(lambda g: g["amount"].sum() >= 600)

print("\ndept_summary:\n", dept_summary)
print("\ncity_dept_summary:\n", city_dept_summary)
print("\nsales_with_group_features:\n", sales_with_group_features)
print("\nlarge_group_only:\n", large_group_only)



dept_summary:
             avg_salary  avg_perf  headcount
department                                 
Data           80000.0       4.5          2
Ops            61000.0       3.9          2
Sales          66000.0       4.1          2

city_dept_summary:
                    avg_salary  total_experience
city   department                              
Berlin Data           80000.0                14
Paris  Sales          66000.0                 9
Rome   Ops            61000.0                10

sales_with_group_features:
    order_id  customer_id   product  channel  amount  quantity order_date  channel_avg_amount  amount_vs_channel_avg
0      1001            1    Laptop      web    1200         1 2025-01-03              416.25                 783.75
1      1002            2     Mouse    store      25         1 2025-01-04               27.50                  -2.50
2      1003            1  Keyboard      web      80         1 2025-01-10              416.25                -336.25
3      100

In [18]:
# Sequencing features inside groups: cumcount / ngroup
sales_seq = sales_df.sort_values(["customer_id", "order_date"]).copy()
sales_seq["order_number_for_customer"] = sales_seq.groupby("customer_id").cumcount() + 1
sales_seq["channel_group_id"] = sales_seq.groupby("channel").ngroup()

print(sales_seq[["order_id", "customer_id", "channel", "order_date", "order_number_for_customer", "channel_group_id"]])

   order_id  customer_id  channel order_date  order_number_for_customer  channel_group_id
0      1001            1      web 2025-01-03                          1                 2
2      1003            1      web 2025-01-10                          2                 2
1      1002            2    store 2025-01-04                          1                 1
5      1006            2      web 2025-02-08                          2                 2
3      1004            3  partner 2025-01-15                          1                 0
7      1008            3  partner 2025-02-20                          2                 0
4      1005            4    store 2025-02-02                          1                 1
6      1007            5      web 2025-02-14                          1                 2


In [19]:
# Suffix handling and cleanup after merge
left_demo = ml_df[["customer_id", "city", "income"]].copy()
right_demo = pd.DataFrame({
    "customer_id": [1, 2, 3, 6],
    "city": ["Berlin", "Paris", "Berlin", "Madrid"],
    "income": [43000, 69000, 51000, 74000]
})

suffix_merged = left_demo.merge(right_demo, on="customer_id", how="left", suffixes=("_left", "_right"))
# cleanup policy: trust left income, keep right city as reference
suffix_merged = suffix_merged.rename(columns={"city_right": "city_reference"}).drop(columns=["income_right"])

print(suffix_merged.head())

   customer_id city_left  income_left city_reference
0            1    Berlin        42000         Berlin
1            2     Paris        68000          Paris
2            3    Berlin        52000         Berlin
3            4      Rome        61000            NaN
4            5     Paris        88000            NaN


## 15. Combining data with concat, merge, and join

These operations look simple, but they create many real bugs. `concat` stacks objects, `merge` joins on key columns, and `join` is convenient for index-based joining. ⚠️ The biggest merge mistake is forgetting key cardinality. If both sides contain repeated keys, a merge can create a row explosion.


In [20]:
vertical_concat = pd.concat(
    [
        sales_df.iloc[:3][["order_id", "customer_id", "amount"]],
        sales_df.iloc[3:6][["order_id", "customer_id", "amount"]],
    ],
    ignore_index=True,
)

horizontal_concat = pd.concat(
    [feature_a_df.set_index("customer_id"), ml_df.set_index("customer_id")[["plan", "churned"]]],
    axis=1,
)

one_to_one_merge = ml_df.merge(feature_a_df, on="customer_id", how="left")
one_to_many_merge = ml_df.merge(feature_b_df, on="customer_id", how="left", suffixes=("", "_order"))

many_left = pd.DataFrame({"key": ["A", "A", "B"], "left_val": [1, 2, 3]})
many_right = pd.DataFrame({"key": ["A", "A", "C"], "right_val": [10, 20, 30]})
many_to_many_merge = many_left.merge(many_right, on="key", how="inner")

joined_on_index = feature_a_df.set_index("customer_id").join(
    ml_df.set_index("customer_id")[["plan", "churned"]],
    how="left"
)

print("\nvertical_concat:\n", vertical_concat)
print("\nhorizontal_concat:\n", horizontal_concat)
print("\none_to_one_merge:\n", one_to_one_merge)
print("\none_to_many_merge:\n", one_to_many_merge)
print("\nmany_to_many_merge:\n", many_to_many_merge)
print("\njoined_on_index:\n", joined_on_index)



vertical_concat:
    order_id  customer_id  amount
0      1001            1    1200
1      1002            2      25
2      1003            1      80
3      1004            3    1250
4      1005            4      30
5      1006            2     300

horizontal_concat:
             region  credit_score        plan  churned
customer_id                                          
1               DE         710.0       basic        0
2               FR         690.0         pro        1
3               DE         730.0       basic        0
4               IT         640.0  enterprise        0
5              NaN           NaN         pro        1
6              NaN           NaN       basic        0

one_to_one_merge:
    customer_id   age  income    city        plan signup_date  churned region  credit_score
0            1  25.0   42000  Berlin       basic  2024-05-01        0     DE         710.0
1            2  34.0   68000   Paris         pro  2024-06-14        1     FR         690.0
2   

?? Merge safety habit: always check row counts before/after, and reason about key uniqueness on both sides before merging.

In [21]:
left_keys = feature_a_df["customer_id"]
right_keys = ml_df["customer_id"]
print("left rows:", len(feature_a_df), "| unique keys:", left_keys.nunique())
print("right rows:", len(ml_df), "| unique keys:", right_keys.nunique())

pre_rows = len(ml_df)
merged_check = ml_df.merge(feature_a_df, on="customer_id", how="left", validate="one_to_one")
post_rows = len(merged_check)
print("rows before:", pre_rows, "rows after:", post_rows)

left rows: 4 | unique keys: 4
right rows: 6 | unique keys: 6
rows before: 6 rows after: 6


In [22]:
# Suffix handling and cleanup after merge
left_demo = ml_df[["customer_id", "city", "income"]].copy()
right_demo = pd.DataFrame({
    "customer_id": [1, 2, 3, 6],
    "city": ["Berlin", "Paris", "Berlin", "Madrid"],
    "income": [43000, 69000, 51000, 74000]
})

suffix_merged = left_demo.merge(right_demo, on="customer_id", how="left", suffixes=("_left", "_right"))
# cleanup policy: trust left income, keep right city as reference
suffix_merged = suffix_merged.rename(columns={"city_right": "city_reference"}).drop(columns=["income_right"])

print(suffix_merged.head())

   customer_id city_left  income_left city_reference
0            1    Berlin        42000         Berlin
1            2     Paris        68000          Paris
2            3    Berlin        52000         Berlin
3            4      Rome        61000            NaN
4            5     Paris        88000            NaN


## 16. Reshaping data

Reshaping changes the layout of a table without changing the underlying information. It is one of the most important pandas skills because real datasets rarely arrive in the shape we want for analysis or plotting.

A helpful way to think about reshaping is:

- **Wide format**: one row can contain several measurement columns such as `salary`, `performance_score`, `q1_sales`, and `q2_sales`.
- **Long format**: the same measurements are stacked into two columns, usually one column that stores the variable name and another that stores the value.

### `melt`: wide to long

`melt` takes multiple columns and turns them into row values. This is useful when several columns represent the same kind of information and should really be treated as one variable.

In the example below, `employee_id` and `name` stay fixed because they identify each employee. The columns `salary` and `performance_score` are unpivoted into two new columns:

- `metric`: stores the original column name
- `value`: stores the value that came from that column

That means each employee appears in multiple rows after the reshape. Use `melt` when column names currently hold data that would be more useful as row values.

### `pivot`: long to wide

`pivot` does the reverse kind of operation. It takes values from one column and spreads them across new columns. In the `monthly_sales` example, the `channel` values (`web`, `store`) become new columns, and their `revenue` values fill the table.

Use `pivot` when each combination of `index` and `columns` points to exactly one value. If there are duplicate combinations, `pivot` will fail because pandas does not know which value to place in the cell.

### `pivot_table`: long to wide with aggregation

`pivot_table` is safer and more flexible than `pivot`. It handles repeated combinations by aggregating them with a function such as `sum`, `mean`, `count`, or `max`.

In the sales example, several rows may belong to the same `product` and `channel`, so `pivot_table(..., aggfunc="sum")` adds them together. `fill_value=0` replaces missing combinations with zero, which often makes result tables easier to read.

A simple rule of thumb is:

- use `pivot` when values are unique
- use `pivot_table` when duplicates may exist or when you want summary statistics

### `stack` and `unstack`

`stack` moves columns into the index, producing a longer result. `unstack` moves an index level back into columns. They are especially useful when working with MultiIndex objects and are conceptually related to `melt` and `pivot`.


In [23]:
monthly_sales = pd.DataFrame(
    {
        "month": ["2025-01", "2025-01", "2025-02", "2025-02"],
        "channel": ["web", "store", "web", "store"],
        "revenue": [1565, 55, 385, 30],
    }
)

melted = employees_df.melt(
    id_vars=["employee_id", "name"],
    value_vars=["salary", "performance_score"],
    var_name="variable",
    value_name="value",
)

pivoted = monthly_sales.pivot(index="month", columns="channel", values="revenue")
pivot_tabled = sales_df.pivot_table(index="product", columns="channel", values="amount", aggfunc="sum", fill_value=0)
stacked = pivoted.stack()
unstacked = stacked.unstack()

print("\nemployees_df:\n", employees_df)
print("\nmelted:\n", melted)
print("\npivoted:\n", pivoted)
print("\npivot_tabled:\n", pivot_tabled)
print("\nstacked:\n", stacked)
print("\nunstacked:\n", unstacked)



employees_df:
    employee_id  name department    city   salary  performance_score  years_experience  is_manager start_date  \
0          101   Ana       Data  Berlin  78000.0                4.7                 6        True 2019-01-15   
1          102   Ben       Data  Berlin  82000.0                4.3                 8       False 2017-07-01   
2          103  Cara      Sales   Paris  65000.0                4.1                 4       False 2021-03-10   
3          104   Dan      Sales   Paris  67000.0                NaN                 5        True 2020-11-05   
4          105   Eli        Ops    Rome      NaN                3.8                 7       False 2018-05-20   
5          106   Fay        Ops    Rome  61000.0                4.0                 3       False 2022-08-14   

  salary_bins exp_quantiles  
0        high           mid  
1        high        senior  
2         low        junior  
3         mid           mid  
4         NaN        senior  
5         low      

### `explode`: one list-like cell into many rows

`explode` is used when a single cell contains multiple values inside a list-like object. Instead of keeping that list in one row, pandas creates a separate row for each item in the list.

In this example, each `record_id` has a list of tags. After `explode("tags")`, every tag gets its own row while `record_id` is repeated to preserve the relationship.

This is especially useful after reading JSON or API data, where columns often contain lists such as:

- tags
- categories
- product attributes
- event participants

Use `ignore_index=True` when you want pandas to create a clean new index after the expansion. Without it, the original row index is repeated for the exploded rows.

A good mental model is: `melt` reshapes multiple columns into rows, while `explode` reshapes multiple values inside one cell into rows.


In [24]:
tag_df = pd.DataFrame({
    "record_id": [1, 2, 3],
    "tags": [["ml", "pandas"], ["sql"], ["ml", "feature-eng", "pipeline"]]
})

exploded_tags = tag_df.explode("tags", ignore_index=True)
print(exploded_tags)

   record_id         tags
0          1           ml
1          1       pandas
2          2          sql
3          3           ml
4          3  feature-eng
5          3     pipeline


## 17. Working with strings

String cleanup is common in imported data. pandas exposes vectorized string methods through the `.str` accessor. Typical tasks include normalizing casing, stripping whitespace, replacing tokens, and extracting structured pieces from messy text.


In [25]:
string_clean = messy_df.copy()
string_clean["name_clean"] = string_clean["raw_name"].str.strip().str.title()
string_clean["city_clean"] = string_clean["raw_city"].str.strip().str.title()
string_clean["segment_clean"] = string_clean["raw_segment"].str.strip().str.lower()

contains_berlin = string_clean["city_clean"].str.contains("Berlin", na=False)
replaced_segment = string_clean["segment_clean"].str.replace("mid-market", "midmarket", regex=False)

email_df = pd.DataFrame({"email": ["ana@company.com", "ben@startup.io", "cara@company.com"]})
email_df["domain"] = email_df["email"].str.extract(r"@(.+)$")

print("\nstring_clean:\n", string_clean)
print("\ncontains_berlin:\n", contains_berlin)
print("\nreplaced_segment:\n", replaced_segment)
print("\nemail_df:\n", email_df)



string_clean:
     raw_name  raw_city raw_segment revenue_text signup_text name_clean city_clean segment_clean
0    alice      berlin  Enterprise         1000  2025-01-10      Alice     Berlin    enterprise
1        BOB    PARIS   enterprise        2,500  01/11/2025        Bob      Paris    enterprise
2     Carla       Rome         SMB         None  2025.01.12      Carla       Rome           smb
3       None     rome         smb           850        None       None       Rome           smb
4      david   BERLIN   Mid-Market        1_200  2025-01-14      David     Berlin    mid-market

contains_berlin:
 0     True
1    False
2    False
3    False
4     True
Name: city_clean, dtype: bool

replaced_segment:
 0    enterprise
1    enterprise
2           smb
3           smb
4     midmarket
Name: segment_clean, dtype: object

email_df:
               email       domain
0   ana@company.com  company.com
1    ben@startup.io   startup.io
2  cara@company.com  company.com


## 18. Working with dates and times

Datetime columns unlock year, month, day, weekday, time differences, sorting, and resampling. For ML work, date columns often become multiple useful features rather than staying as raw strings.


In [26]:
sales_time = sales_df.copy()
sales_time["order_year"] = sales_time["order_date"].dt.year
sales_time["order_month"] = sales_time["order_date"].dt.month
sales_time["weekday"] = sales_time["order_date"].dt.day_name()
sales_time["days_since_first_order"] = (sales_time["order_date"] - sales_time["order_date"].min()).dt.days

metrics_time = metrics_df.set_index("date")
weekly_resample = metrics_time.resample("W").sum()
rolling_visits = metrics_time["visits"].rolling(window=3).mean()

print("\nsales_time:\n", sales_time)
print("\nweekly_resample:\n", weekly_resample)
print("\nrolling_visits:\n", rolling_visits)



sales_time:
    order_id  customer_id   product  channel  amount  quantity order_date  order_year  order_month    weekday  \
0      1001            1    Laptop      web    1200         1 2025-01-03        2025            1     Friday   
1      1002            2     Mouse    store      25         1 2025-01-04        2025            1   Saturday   
2      1003            1  Keyboard      web      80         1 2025-01-10        2025            1     Friday   
3      1004            3    Laptop  partner    1250         1 2025-01-15        2025            1  Wednesday   
4      1005            4     Mouse    store      30         2 2025-02-02        2025            2     Sunday   
5      1006            2   Monitor      web     300         1 2025-02-08        2025            2   Saturday   
6      1007            5  Keyboard      web      85         1 2025-02-14        2025            2     Friday   
7      1008            3   Monitor  partner     320         1 2025-02-20        2025      

?? Datetime parsing can silently coerce bad values to `NaT` with `errors='coerce'`. Always measure parse failure rate.

In [27]:
raw_dates = pd.Series(["2025-01-01", "01/02/2025", "bad-date", "2025.01.04"])
parsed_dates = pd.to_datetime(raw_dates, errors="coerce")

failure_rate = parsed_dates.isna().mean()
print(pd.DataFrame({"raw": raw_dates, "parsed": parsed_dates}))
print("parse failure rate:", round(failure_rate, 3))

          raw     parsed
0  2025-01-01 2025-01-01
1  01/02/2025        NaT
2    bad-date        NaT
3  2025.01.04        NaT
parse failure rate: 0.75


## 19. Duplicates and data cleaning workflows

Real datasets often arrive with duplicated rows, inconsistent casing, whitespace issues, messy column names, and mixed formatting rules. A cleaning workflow usually does a few small, explicit transformations rather than one giant mysterious function.


In [28]:
dirty_sales = pd.concat([sales_df, sales_df.iloc[[0]]], ignore_index=True)
duplicate_mask = dirty_sales.duplicated()
deduped_sales = dirty_sales.drop_duplicates()

cleaned_messy = messy_df.copy()
cleaned_messy.columns = cleaned_messy.columns.str.strip().str.lower()
cleaned_messy["raw_name"] = cleaned_messy["raw_name"].str.strip().str.title()
cleaned_messy["raw_city"] = cleaned_messy["raw_city"].str.strip().str.title()
cleaned_messy["raw_segment"] = cleaned_messy["raw_segment"].str.strip().str.lower().replace({"smb": "SMB", "enterprise": "Enterprise"})

outlier_filtered = sales_df[sales_df["amount"] <= sales_df["amount"].quantile(0.95)]

print("\nduplicate_mask.sum():\n", duplicate_mask.sum())
print("\ndeduped_sales.shape:\n", deduped_sales.shape)
print("\ncleaned_messy:\n", cleaned_messy)
print("\noutlier_filtered:\n", outlier_filtered)



duplicate_mask.sum():
 1

deduped_sales.shape:
 (8, 7)

cleaned_messy:
   raw_name raw_city raw_segment revenue_text signup_text
0    Alice   Berlin  Enterprise         1000  2025-01-10
1      Bob    Paris  Enterprise        2,500  01/11/2025
2    Carla     Rome         SMB         None  2025.01.12
3     None     Rome         SMB          850        None
4    David   Berlin  mid-market        1_200  2025-01-14

outlier_filtered:
    order_id  customer_id   product  channel  amount  quantity order_date
0      1001            1    Laptop      web    1200         1 2025-01-03
1      1002            2     Mouse    store      25         1 2025-01-04
2      1003            1  Keyboard      web      80         1 2025-01-10
4      1005            4     Mouse    store      30         2 2025-02-02
5      1006            2   Monitor      web     300         1 2025-02-08
6      1007            5  Keyboard      web      85         1 2025-02-14
7      1008            3   Monitor  partner     320   

## 20. MultiIndex basics

A MultiIndex lets you represent multiple keys on rows or columns. It is useful for grouped summaries and reshaped tables, but it can become cumbersome if overused. Use it as your last resort


In [29]:
multi_summary = sales_df.groupby(["channel", "product"]).agg(
    total_amount=("amount", "sum"),
    order_count=("order_id", "count"),
)

sorted_multi = multi_summary.sort_index()
selected_multi = multi_summary.loc[("web", slice(None)), :]
reset_multi = multi_summary.reset_index()

print("\nmulti_summary:\n", multi_summary)
print("\nsorted_multi:\n", sorted_multi)
print("\nselected_multi:\n", selected_multi)
print("\nreset_multi:\n", reset_multi)



multi_summary:
                   total_amount  order_count
channel product                            
partner Laptop            1250            1
        Monitor            320            1
store   Mouse               55            2
web     Keyboard           165            2
        Laptop            1200            1
        Monitor            300            1

sorted_multi:
                   total_amount  order_count
channel product                            
partner Laptop            1250            1
        Monitor            320            1
store   Mouse               55            2
web     Keyboard           165            2
        Laptop            1200            1
        Monitor            300            1

selected_multi:
                   total_amount  order_count
channel product                            
web     Keyboard           165            2
        Laptop            1200            1
        Monitor            300            1

reset_multi:
    channel

A common question with MultiIndex objects is how to select part of the table. You can pass a tuple to `.loc` to target multiple index levels.


In [ ]:
web_only = sorted_multi.loc["web"]
web_keyboard = sorted_multi.loc[("web", "Keyboard"), :]

print("\nweb_only:\n", web_only)
print("\nweb_keyboard:\n", web_keyboard)


web_only:
           total_amount  order_count
product                            
Keyboard           165            2
Laptop            1200            1
Monitor            300            1

web_keyboard:
 total_amount    165
order_count       2
Name: (web, Keyboard), dtype: int64

channel_slice:
                   total_amount  order_count
channel product                            
partner Laptop            1250            1
        Monitor            320            1
store   Mouse               55            2
web     Keyboard           165            2
        Laptop            1200            1
        Monitor            300            1


## 21. Apply, map, transform, and vectorization

These tools overlap, but they are not interchangeable. `map` is great for value mapping in a Series, `apply` is flexible but often slower, `transform` is ideal for group-level features aligned back to each row, and vectorization is usually the first choice when possible.


In [31]:
mapped_department = employees_df["department"].map({"Data": 1, "Sales": 2, "Ops": 3})
applied_name_length = employees_df["name"].apply(len)
rowwise_apply = employees_df.apply(lambda row: f"{row['name']} ({row['department']})", axis=1)
vectorized_salary_ratio = employees_df["salary"] / employees_df["salary"].mean()
group_transform = sales_df.groupby("product")["amount"].transform(lambda s: s / s.sum())

print("\nmapped_department:\n", mapped_department)
print("\napplied_name_length:\n", applied_name_length)
print("\nrowwise_apply:\n", rowwise_apply)
print("\nvectorized_salary_ratio:\n", vectorized_salary_ratio)
print("\ngroup_transform:\n", group_transform)



mapped_department:
 0    1
1    1
2    2
3    2
4    3
5    3
Name: department, dtype: int64

applied_name_length:
 0    3
1    3
2    4
3    3
4    3
5    3
Name: name, dtype: int64

rowwise_apply:
 0      Ana (Data)
1      Ben (Data)
2    Cara (Sales)
3     Dan (Sales)
4       Eli (Ops)
5       Fay (Ops)
dtype: object

vectorized_salary_ratio:
 0    1.104816
1    1.161473
2    0.920680
3    0.949008
4         NaN
5    0.864023
Name: salary, dtype: float64

group_transform:
 0    0.489796
1    0.454545
2    0.484848
3    0.510204
4    0.545455
5    0.483871
6    0.515152
7    0.516129
Name: amount, dtype: float64


`pipe` helps keep transformation chains readable by making each step explicit and composable.

In [ ]:
def add_income_band(df):
    out = df.copy()
    out["income_band"] = pd.cut(out["income"], bins=[0, 50000, 70000, 1e9], labels=["low", "mid", "high"]) 
    return out

def add_tenure_bucket(df):
    out = df.copy()
    out["tenure_bucket"] = pd.qcut(out["age"], q=3, labels=["young", "mid", "senior"])
    return out

piped = (
    ml_df
    .pipe(lambda d: d.assign(age=d["age"].fillna(d["age"].median())))
    .pipe(add_income_band)
    .pipe(add_tenure_bucket)
)

print("\nml_df: \n", ml_df)
print("\n")
print(piped[["customer_id", "age", "income", "income_band", "tenure_bucket"]])


ml_df: 
    customer_id   age  income    city        plan signup_date  churned
0            1  25.0   42000  Berlin       basic  2024-05-01        0
1            2  34.0   68000   Paris         pro  2024-06-14        1
2            3  29.0   52000  Berlin       basic  2024-07-20        0
3            4   NaN   61000    Rome  enterprise  2024-08-03        0
4            5  42.0   88000   Paris         pro  2024-09-15        1
5            6  37.0   73000    None       basic  2024-10-01        0


   customer_id   age  income income_band tenure_bucket
0            1  25.0   42000         low         young
1            2  34.0   68000         mid           mid
2            3  29.0   52000         mid         young
3            4  34.0   61000         mid           mid
4            5  42.0   88000        high        senior
5            6  37.0   73000        high        senior


## 22. Window functions and cumulative operations

Window operations preserve row order and compute summaries over rolling or expanding slices of the data. These patterns are useful for time-based metrics, smoothing noisy signals, running totals, and trend features.


In [47]:
metrics_window = metrics_df.copy().set_index("date")
metrics_window["rolling_visits_3"] = metrics_window["visits"].rolling(3).mean()
metrics_window["expanding_signup_avg"] = metrics_window["signups"].expanding().mean()
metrics_window["cumulative_visits"] = metrics_window["visits"].cumsum()
metrics_window["cummax_visits"] = metrics_window["visits"].cummax()

print("metrics_window:\n", metrics_window)


metrics_window:
             visits  signups  rolling_visits_3  expanding_signup_avg  cumulative_visits  cummax_visits
date                                                                                                 
2025-03-01     120        8               NaN              8.000000                120            120
2025-03-02     135       10               NaN              9.000000                255            135
2025-03-03     128        9        127.666667              9.000000                383            135
2025-03-04     150       11        137.666667              9.500000                533            150
2025-03-05     165       14        147.666667             10.400000                698            165
2025-03-06     160       13        158.333333             10.833333                858            165
2025-03-07     172       15        165.666667             11.428571               1030            172
2025-03-08     180       16        170.666667             12.0000

## 23. Performance and memory awareness

Most pandas performance wins come from good habits, not micro-optimizations: load only needed columns, specify dtypes when possible, prefer vectorized operations, use `category` for repeated strings, and be skeptical of heavy row-wise `apply`.


In [34]:
memory_report = ml_df.memory_usage(deep=True).sort_values(ascending=False)
city_as_object = ml_df["city"].astype("object")
city_as_category = ml_df["city"].astype("category")

memory_compare = pd.Series(
    {
        "city_object_bytes": city_as_object.memory_usage(deep=True),
        "city_category_bytes": city_as_category.memory_usage(deep=True),
    }
)

vectorized_feature = sales_df.assign(amount_per_item=sales_df["amount"] / sales_df["quantity"])

print("\nmemory_report:\n", memory_report)
print("\nmemory_compare:\n", memory_compare)
print("\nvectorized_feature:\n", vectorized_feature.head())



memory_report:
 plan           325
city           295
Index          132
customer_id     48
age             48
income          48
signup_date     48
churned         48
dtype: int64

memory_compare:
 city_object_bytes      427
city_category_bytes    408
dtype: int64

vectorized_feature:
    order_id  customer_id   product  channel  amount  quantity order_date  amount_per_item
0      1001            1    Laptop      web    1200         1 2025-01-03           1200.0
1      1002            2     Mouse    store      25         1 2025-01-04             25.0
2      1003            1  Keyboard      web      80         1 2025-01-10             80.0
3      1004            3    Laptop  partner    1250         1 2025-01-15           1250.0
4      1005            4     Mouse    store      30         2 2025-02-02             15.0


## 24. ML-oriented preprocessing patterns

A common pandas goal is to produce a clean table that can be passed to modeling code. Typical steps are choosing target and feature columns, cleaning or imputing missing values, encoding categoricals, deriving date features, and avoiding leakage columns.


In [35]:
ml_ready = ml_df.copy()
ml_ready["age"] = ml_ready["age"].fillna(ml_ready["age"].median())
ml_ready["city"] = ml_ready["city"].fillna("Unknown")
ml_ready["signup_month"] = ml_ready["signup_date"].dt.month
ml_ready["is_high_income"] = (ml_ready["income"] >= 70000).astype(int)

class_balance = ml_ready["churned"].value_counts(normalize=True)
X_base = ml_ready.drop(columns=["customer_id", "churned"])
y = ml_ready["churned"]
X_encoded = pd.get_dummies(X_base, columns=["city", "plan"], drop_first=False)
numeric_cols = ["age", "income"]
X_encoded[numeric_cols] = (X_encoded[numeric_cols] - X_encoded[numeric_cols].mean()) / X_encoded[numeric_cols].std()

print("\nX_encoded:\n", X_encoded)
print("\ny:\n", y)
print("\nclass_balance:\n", class_balance)



X_encoded:
         age    income signup_date  signup_month  is_high_income  city_Berlin  city_Paris  city_Rome  city_Unknown  \
0 -1.426608 -1.359165  2024-05-01             5               0         True       False      False         False   
1  0.083918  0.247121  2024-06-14             6               0        False        True      False         False   
2 -0.755263 -0.741362  2024-07-20             7               0         True       False      False         False   
3  0.083918 -0.185341  2024-08-03             8               0        False       False       True         False   
4  1.426608  1.482725  2024-09-15             9               1        False        True      False         False   
5  0.587427  0.556022  2024-10-01            10               1        False       False      False          True   

   plan_basic  plan_enterprise  plan_pro  
0        True            False     False  
1       False            False      True  
2        True            False     Fal

ML engineer mindset in pandas: guard against leakage, keep train/inference schemas consistent, and make imbalance checks explicit.

In [36]:
# train/test mindset before fit-time transformations
split_idx = int(len(ml_ready) * 0.67)
train_df = ml_ready.iloc[:split_idx].copy()
test_df = ml_ready.iloc[split_idx:].copy()

X_train_base = train_df.drop(columns=["churned", "customer_id"])
X_test_base = test_df.drop(columns=["churned", "customer_id"])
y_train = train_df["churned"]
y_test = test_df["churned"]

print("train/test rows:", len(train_df), len(test_df))
print("\ntrain class ratio:\n", y_train.value_counts(normalize=True))

train/test rows: 4 2

train class ratio:
 churned
0    0.75
1    0.25
Name: proportion, dtype: float64


In [37]:
# consistent one-hot columns between train and inference
X_train_enc = pd.get_dummies(X_train_base, columns=["city", "plan"], drop_first=False)
X_test_enc = pd.get_dummies(X_test_base, columns=["city", "plan"], drop_first=False)

# align test to train schema
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print("X_train columns:", len(X_train_enc.columns))
print("X_test columns after align:", len(X_test_enc.columns))
print("same columns:", X_train_enc.columns.equals(X_test_enc.columns))

X_train columns: 11
X_test columns after align: 11
same columns: True


In [38]:
# leakage warning demo: never include target-derived columns in X
leaky_df = ml_ready.copy()
leaky_df["target_copy"] = leaky_df["churned"]  # intentionally leaky feature

safe_features = [c for c in leaky_df.columns if c not in ["churned", "target_copy", "customer_id"]]
X_safe = pd.get_dummies(leaky_df[safe_features], columns=["city", "plan"], drop_first=False)

print("safe feature columns:", X_safe.columns.tolist())

safe feature columns: ['age', 'income', 'signup_date', 'signup_month', 'is_high_income', 'city_Berlin', 'city_Paris', 'city_Rome', 'city_Unknown', 'plan_basic', 'plan_enterprise', 'plan_pro']


## 25. Common pitfalls and best practices

A few pandas mistakes cause outsized pain: SettingWithCopy or chained assignment, index alignment surprises, merge row explosion, object dtype drift, and overusing `apply`. Best practice is usually to inspect early, make dtypes explicit, clean incrementally, merge carefully, and keep transformations readable.


In [39]:
left = pd.Series([100, 200], index=["x", "y"])
right = pd.Series([1, 2], index=["y", "z"])
alignment_surprise = left + right

safe_assignment_df = employees_df.copy()
safe_assignment_df.loc[safe_assignment_df["salary"].isna(), "salary"] = safe_assignment_df["salary"].median()

merge_explosion_rows = many_to_many_merge.shape[0]

print("\nalignment_surprise:\n", alignment_surprise)
print("safe_assignment_df[[\"name\", \"salary\"]]:\n", safe_assignment_df[["name", "salary"]])
print("\nmerge_explosion_rows:\n", merge_explosion_rows)



alignment_surprise:
 x      NaN
y    201.0
z      NaN
dtype: float64
safe_assignment_df[["name", "salary"]]:
    name   salary
0   Ana  78000.0
1   Ben  82000.0
2  Cara  65000.0
3   Dan  67000.0
4   Eli  67000.0
5   Fay  61000.0

merge_explosion_rows:
 4


?? If you remember only four checks before shipping a dataframe transformation, remember these: row count, key uniqueness, dtype sanity, and null profile.

In [40]:
pre_rows = len(ml_df)
post_rows = len(ml_df.merge(feature_b_df, on="customer_id", how="left"))
print("merge row count change:", pre_rows, "->", post_rows)
print("feature_b unique keys:", feature_b_df["customer_id"].nunique(), "of", len(feature_b_df))

dtype_check = ml_df.dtypes.astype(str)
null_check = ml_df.isna().sum()
print("\ndtypes:\n", dtype_check)
print("\nnull counts:\n", null_check)

merge row count change: 6 -> 7
feature_b unique keys: 3 of 4

dtypes:
 customer_id             int64
age                   float64
income                  int64
city                   object
plan                   object
signup_date    datetime64[ns]
churned                 int64
dtype: object

null counts:
 customer_id    0
age            1
income         0
city           1
plan           0
signup_date    0
churned        0
dtype: int64


## 26. End-to-end realistic workflow

This final section ties together inspection, cleaning, missing-value handling, feature creation, grouping, joining, and model-table preparation. The point is not to build a full production pipeline in one cell. The point is to show what a realistic pandas preparation flow looks like when the input data is a bit messy.


In [41]:
workflow_df = pd.DataFrame(
    {
        "customer_id": [1, 2, 3, 4, 5],
        "name": [" alice ", "BOB", "Carla", "david ", None],
        "city": [" berlin", "PARIS ", "Rome", "rome ", None],
        "plan": ["basic", "pro", "basic", "enterprise", "pro"],
        "income_text": ["42000", "68000", "52000", None, "88000"],
        "signup_text": ["2024-05-01", "2024-06-14", "2024-07-20", "2024-08-03", "2024-09-15"],
        "churned": [0, 1, 0, 0, 1],
    }
)

workflow_profile = {
    "shape": workflow_df.shape,
    "null_counts": workflow_df.isna().sum().to_dict(),
    "dtypes": workflow_df.dtypes.astype(str).to_dict(),
}

workflow_df["name"] = workflow_df["name"].str.strip().str.title().fillna("Unknown")
workflow_df["city"] = workflow_df["city"].str.strip().str.title().fillna("Unknown")
workflow_df["income"] = pd.to_numeric(workflow_df["income_text"], errors="coerce")
workflow_df["signup_date"] = pd.to_datetime(workflow_df["signup_text"], errors="coerce")
workflow_df["income"] = workflow_df["income"].fillna(workflow_df["income"].median())
workflow_df["signup_month"] = workflow_df["signup_date"].dt.month
workflow_df["name_length"] = workflow_df["name"].str.len()
workflow_df["is_high_income"] = (workflow_df["income"] >= 70000).astype(int)

workflow_df = workflow_df.merge(feature_a_df, on="customer_id", how="left")
customer_sales = sales_df.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    avg_order_value=("amount", "mean"),
    order_count=("order_id", "count"),
)
workflow_df = workflow_df.merge(customer_sales, on="customer_id", how="left")
workflow_df[["total_spend", "avg_order_value", "order_count"]] = workflow_df[["total_spend", "avg_order_value", "order_count"]].fillna(0)

model_ready = pd.get_dummies(
    workflow_df[
        [
            "income",
            "signup_month",
            "name_length",
            "is_high_income",
            "credit_score",
            "total_spend",
            "avg_order_value",
            "order_count",
            "city",
            "plan",
            "region",
        ]
    ],
    columns=["city", "plan", "region"],
    drop_first=False,
)

target = workflow_df["churned"]

print("\nworkflow_profile:\n", workflow_profile)
print("\nworkflow_df:\n", workflow_df)
print("\nmodel_ready:\n", model_ready)
print("\ntarget:\n", target)



workflow_profile:
 {'shape': (5, 7), 'null_counts': {'customer_id': 0, 'name': 1, 'city': 1, 'plan': 0, 'income_text': 1, 'signup_text': 0, 'churned': 0}, 'dtypes': {'customer_id': 'int64', 'name': 'object', 'city': 'object', 'plan': 'object', 'income_text': 'object', 'signup_text': 'object', 'churned': 'int64'}}

workflow_df:
    customer_id     name     city        plan income_text signup_text  churned   income signup_date  signup_month  \
0            1    Alice   Berlin       basic       42000  2024-05-01        0  42000.0  2024-05-01             5   
1            2      Bob    Paris         pro       68000  2024-06-14        1  68000.0  2024-06-14             6   
2            3    Carla     Rome       basic       52000  2024-07-20        0  52000.0  2024-07-20             7   
3            4    David     Rome  enterprise        None  2024-08-03        0  60000.0  2024-08-03             8   
4            5  Unknown  Unknown         pro       88000  2024-09-15        1  88000.0  2